# 영상 해상도 축소 (1920x1080 → 960x540)

Re-ID 파이프라인 처리 속도 향상을 위해 원본 영상을 절반 크기로 변환합니다.

| 항목 | 내용 |
|---|---|
| 입력 | `data/*.avi` (1920x1080) |
| 출력 | `data_960/*.avi` (960x540) |
| 코덱 | libx264, CRF 18 (고품질) |
| 예상 효과 | 파일 크기 ~75% 감소, 처리 속도 ~3배 향상 |

> 원본 파일은 변경하지 않습니다.

In [ ]:
import subprocess, os, pathlib, time

# ── 경로 설정 ──────────────────────────────────────────────────────────────
_here     = pathlib.Path(globals().get('__vsc_ipynb_file__', __file__) if '__file__' in dir() else '.').resolve()
ROOT      = _here.parents[1] if 'notebooks' in str(_here) else pathlib.Path('.').resolve()

DATA_DIR  = ROOT / 'data'       # 원본 영상 폴더
OUT_DIR   = ROOT / 'data_960'   # 출력 폴더
WIDTH, HEIGHT = 960, 540
CRF       = 18                  # 품질 (낮을수록 고품질·큰 파일, 18=고품질)

OUT_DIR.mkdir(exist_ok=True)

# FFmpeg 설치 확인
result = subprocess.run(['ffmpeg', '-version'], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError('FFmpeg가 설치되어 있지 않습니다. https://ffmpeg.org/download.html')

print(f'입력 폴더: {DATA_DIR}')
print(f'출력 폴더: {OUT_DIR}')
print(f'목표 해상도: {WIDTH}x{HEIGHT}  CRF={CRF}')

videos = sorted(DATA_DIR.glob('*.avi'))
print(f'\n처리 대상: {len(videos)}개')
for v in videos:
    print(f'  {v.name}  ({v.stat().st_size / 1024**2:.0f} MB)')

In [ ]:
import cv2

print('── 원본 영상 정보 ──────────────────────────────────')
for v in videos:
    cap = cv2.VideoCapture(str(v))
    w   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    n   = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    mb  = v.stat().st_size / 1024**2
    print(f'  {v.name}: {w}x{h}  {fps:.1f}fps  {n:,}프레임  {mb:.0f}MB')

In [ ]:
total_start = time.time()
results = []

for v in videos:
    dst = OUT_DIR / v.name

    if dst.exists():
        print(f'⏭  건너뜀 (이미 존재): {v.name}')
        results.append((v.name, v.stat().st_size, dst.stat().st_size, 0))
        continue

    print(f'▶  변환 중: {v.name} ...', end='', flush=True)
    t0  = time.time()

    proc = subprocess.run(
        [
            'ffmpeg', '-y',
            '-i',  str(v),
            '-vf', f'scale={WIDTH}:{HEIGHT}',
            '-c:v', 'libx264',
            '-crf', str(CRF),
            '-preset', 'fast',      # 인코딩 속도 우선
            '-c:a', 'copy',
            str(dst),
        ],
        capture_output=True, text=True
    )

    elapsed = time.time() - t0

    if proc.returncode != 0:
        print(f'  ✗ 실패')
        print(proc.stderr[-500:])
        continue

    orig_mb = v.stat().st_size   / 1024**2
    new_mb  = dst.stat().st_size / 1024**2
    ratio   = new_mb / orig_mb * 100
    print(f'  완료  {orig_mb:.0f}MB → {new_mb:.0f}MB ({ratio:.0f}%)  {elapsed:.0f}초')
    results.append((v.name, v.stat().st_size, dst.stat().st_size, elapsed))

total_elapsed = time.time() - total_start
print(f'\n총 소요: {total_elapsed/60:.1f}분')

In [ ]:
print('── 변환 결과 요약 ──────────────────────────────────')
total_orig = total_new = 0
for name, orig, new, t in results:
    total_orig += orig
    total_new  += new
    print(f'  {name}: {orig/1024**2:.0f}MB → {new/1024**2:.0f}MB  ({new/orig*100:.0f}%)')

print(f'\n합계: {total_orig/1024**2:.0f}MB → {total_new/1024**2:.0f}MB  ({total_new/total_orig*100:.0f}%)')
print(f'절감: {(total_orig - total_new)/1024**2:.0f}MB')
print(f'\n출력 폴더: {OUT_DIR}')
print('\nrun_pipeline_*.ipynb 에서 DATA_DIR 경로를 data_960/ 으로 변경하세요.')